In [1]:
import os

In [2]:
%pwd

'c:\\Users\\mnsnn\\Documents\\Learn Advanced\\ML - MLOps\\datascienceproject\\research'

In [3]:
os.chdir('../')
%pwd

'c:\\Users\\mnsnn\\Documents\\Learn Advanced\\ML - MLOps\\datascienceproject'

In [ ]:
# config_entity.py
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_url: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
# configuration.py
from src.datascienceproject.constants import CONFIG_FILE_PATH
from src.datascienceproject.constants import PARAMS_FILE_PATH
from src.datascienceproject.constants import SCHEMA_FILE_PATH
from src.datascienceproject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    
	def __init__(self, 
			config_file_path = CONFIG_FILE_PATH,
			params_file_path = PARAMS_FILE_PATH,
			schema_file_path = SCHEMA_FILE_PATH
			):
		
		self.config = read_yaml(config_file_path)
		self.params = read_yaml(params_file_path)
		self.schema = read_yaml(schema_file_path)

		# creating the artifacts folder
		create_directories([self.config.artifacts_root])

	def get_data_ingestion_config(self) -> DataIngestionConfig:
		config = self.config.data_ingestion

		# creating data ingestion artifact dir
		create_directories([config.root_dir])

		data_ingestion_config = DataIngestionConfig(
			root_dir = config.root_dir,
			source_url=config.source_url,
			local_data_file = config.local_data_file,
			unzip_dir = config.unzip_dir
		)

		return data_ingestion_config

In [10]:
# data_ingestion.py
import os
import urllib.request as request
from src.datascienceproject import logger
import zipfile

class DataIngestion:
	def __init__(self, config:DataIngestionConfig):
		self.config = config
		
	def download_file(self):

		"Downloading a zipfile"

		if not os.path.exists(self.config.local_data_file):

			filename, headers = request.urlretrieve(
				url = self.config.source_url,
				filename = self.config.local_data_file
			)
			logger.info(f"{filename}: Downloaded Successfully")
		else:
			logger.info(f"File already exists")

	def extract_zip_file(self):

		"Extracting a zipfile"
		
		try:
			unzip_path = self.config.unzip_dir
			os.makedirs(unzip_path, exist_ok=True)
			with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
				zip_ref.extractall(unzip_path)
			logger.info(f"Zipfile extracted to: {unzip_path}")
		except:
			logger.info(f"Failed to extract Zipfile to: {unzip_path}")

In [11]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

2026-05-25 21:46:20,057 : INFO : YAML file config\config.yaml loaded successfully
2026-05-25 21:46:20,059 : INFO : YAML file params.yaml loaded successfully
2026-05-25 21:46:20,060 : INFO : YAML file schema.yaml loaded successfully
2026-05-25 21:46:20,062 : INFO : Created directory at artifacts
2026-05-25 21:46:20,062 : INFO : Created directory at artifacts/data_ingestion
2026-05-25 21:46:20,386 : INFO : artifacts/data_ingestion/winequality-data.zip: Downloaded Successfully
2026-05-25 21:46:20,389 : INFO : Zipfile extracted to: artifacts/data_ingestion
